# Tier-1 LLM Router — Embeddings + GBM + cost-aware routing

Run on a free Colab/Kaggle **GPU** runtime. Upload `train.csv`, `test.csv`,
`sample_submission.csv` (or mount them), then run top-to-bottom. Produces
`submission.csv`. This mirrors `src/route.py` but uses real sentence-embeddings.

**Backbone swap is the only real change vs. the local smoke test.** Try a few
embedding models and keep the best validation reward.

In [ ]:
import importlib.util, subprocess, sys
for _pkg, _pip in [("sentence_transformers", "sentence-transformers"), ("lightgbm", "lightgbm")]:
    if importlib.util.find_spec(_pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _pip], check=False)

In [ ]:
import time, glob, os, numpy as np, pandas as pd
import torch

def pick_device():
    """Use CUDA only if the installed torch actually has kernels for this GPU.
    Kaggle sometimes assigns a Tesla P100 (sm_60) that the shipped torch build
    does not support -> falls back to CPU instead of crashing mid-run."""
    if not torch.cuda.is_available():
        return "cpu"
    try:
        cap = "sm_%d%d" % torch.cuda.get_device_capability(0)
        archs = torch.cuda.get_arch_list()
        if archs and cap not in archs:
            print(f"[device] GPU {cap} not in torch arch list {archs} -> CPU")
            return "cpu"
        torch.zeros(8, device="cuda").sum().item()   # real probe
        return "cuda"
    except Exception as e:
        print(f"[device] CUDA probe failed ({e}) -> CPU")
        return "cpu"

DEVICE = pick_device()

# Auto-detect data dir: Kaggle attaches datasets under /kaggle/input/<slug>/.
_hits = glob.glob("/kaggle/input/*/train.csv") + glob.glob("./train.csv") + glob.glob("./data/train.csv")
DATA = os.path.dirname(_hits[0]) if _hits else "."   # dir holding the 3 csvs
EMB_MODEL = "BAAI/bge-m3"           # try also: intfloat/e5-large-v2, Alibaba-NLP/gte-large-en-v1.5
MAX_TOKENS = 1024                   # truncate long queries (length != difficulty)
SEED, VAL_FRAC = 42, 0.15
PERF_WEIGHT, COST_WEIGHT = 0.85, 0.15

MODELS = list("ABCDEFGHIJK")
PERF_COLS = [f"Model_{m}_performance" for m in MODELS]
COST_COLS = [f"Model_{m}_cost" for m in MODELS]
print("cuda:", torch.cuda.is_available(), "| device:", DEVICE)


In [ ]:
train = pd.read_csv(f"{DATA}/train.csv"); test = pd.read_csv(f"{DATA}/test.csv")
perf = train[PERF_COLS].to_numpy("float32"); cost = train[COST_COLS].to_numpy("float32")
CMAX = float(cost.max())
print("train", train.shape, "test", test.shape, "C_max", CMAX)

In [ ]:
from sentence_transformers import SentenceTransformer
st = SentenceTransformer(EMB_MODEL, device=DEVICE)
st.max_seq_length = MAX_TOKENS
def embed(texts):
    return st.encode(list(texts.astype(str)), batch_size=64, show_progress_bar=True,
                     convert_to_numpy=True, normalize_embeddings=True).astype("float32")
t0 = time.time()
Xall = embed(train["query"]); Xtest = embed(test["query"])
print("embedded", Xall.shape, Xtest.shape, f"{time.time()-t0:.1f}s")
# Optional: np.save("emb_train.npy", Xall); np.save("emb_test.npy", Xtest)


In [ ]:
def reward_from_choice(P, C, ch, cmax):
    r = np.arange(len(ch))
    return PERF_WEIGHT*P[r, ch].mean() - COST_WEIGHT*(C[r, ch].mean()/cmax)
def oracle_choice(P, C, cmax):
    return (PERF_WEIGHT*P - (COST_WEIGHT/cmax)*C).argmax(1)
def route(ph, ch_, a):
    return (ph - a*ch_).argmax(1)

In [ ]:
try:
    import lightgbm as lgb
    def mk(): return lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, num_leaves=63,
                                       subsample=0.8, colsample_bytree=0.8, random_state=SEED, n_jobs=-1)
except ImportError:
    from sklearn.ensemble import HistGradientBoostingRegressor
    def mk(): return HistGradientBoostingRegressor(max_iter=500, learning_rate=0.05,
                                                   max_leaf_nodes=63, early_stopping=True, random_state=SEED)
def fit_pred(Xtr, Ytr, Xp):
    out = np.zeros((Xp.shape[0], Ytr.shape[1]), "float32")
    for j in range(Ytr.shape[1]):
        m = mk(); m.fit(Xtr, Ytr[:, j]); out[:, j] = m.predict(Xp)
    return out

In [ ]:
rng = np.random.default_rng(SEED); idx = rng.permutation(len(train)); nv = int(len(train)*VAL_FRAC)
vi, ti = idx[:nv], idx[nv:]
ph_val = fit_pred(Xall[ti], perf[ti], Xall[vi])
ch_val = fit_pred(Xall[ti], cost[ti], Xall[vi])

In [ ]:
base = COST_WEIGHT/(PERF_WEIGHT*CMAX)
grid = np.r_[0.0, np.geomspace(base*0.01, base*100, 40)]
alpha = max(grid, key=lambda a: reward_from_choice(perf[vi], cost[vi], route(ph_val, ch_val, a), CMAX))
r_val = reward_from_choice(perf[vi], cost[vi], route(ph_val, ch_val, alpha), CMAX)
oracle = reward_from_choice(perf[vi], cost[vi], oracle_choice(perf[vi], cost[vi], CMAX), CMAX)
const = max(reward_from_choice(perf[vi], cost[vi], np.full(nv, j), CMAX) for j in range(11))
print(f"VAL reward: const={const:.4f}  router={r_val:.4f}  oracle={oracle:.4f}  "
      f"(headroom captured {(r_val-const)/(oracle-const):.1%}, alpha={alpha:.4g})")

In [ ]:
ph_te = fit_pred(Xall, perf, Xtest); ch_te = fit_pred(Xall, cost, Xtest)
choice = route(ph_te, ch_te, alpha)
sub = pd.DataFrame({"ID": test["ID"], "pred_model": [f"Model_{MODELS[i]}" for i in choice]})
sub.to_csv("submission.csv", index=False)
print(sub["pred_model"].value_counts()); print("wrote submission.csv")